In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        (os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# installing the ultralytics package
!pip install -q -U ultralytics

In [ ]:
# importing libraries
import hashlib
import json
import re
import shutil
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import ultralytics
import yaml
from PIL import Image as PILImage
from PIL import ImageDraw, ImageOps
from ultralytics import YOLO

warnings.filterwarnings("ignore")

In [ ]:
# checking the GPU and library versions
assert torch.cuda.is_available(), "No GPU found. Session options -> Accelerator -> GPU T4 x2."
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)

In [ ]:
# setting paths and training options
SOURCE_DIR = Path("/kaggle/input/datasets/ternabartholomew/recycling-detection-yolo-v3")
DATA_DIR = Path("/kaggle/working/dataset")
RUNS_DIR = Path("/kaggle/working/runs")
DATA_YAML = DATA_DIR / "data.yaml"
TRAIN_LIST = DATA_DIR / "train_weighted.txt"
TRAIN_YAML = DATA_DIR / "data_train.yaml"
CHANGE_LOG = Path("/kaggle/working/dataset_changes.csv")

MODEL = "yolo11s.pt"
IMG_SIZE = 960
BATCH_SIZE = 8
EPOCHS = 100               
PATIENCE = 30              
SCALE = 0.5                
MAX_DET = 600
WORKERS = 4
SEED = 0
USE_BOTH_GPUS = False
DEVICE = [0, 1] if USE_BOTH_GPUS else 0

FREEZE = 0                 
LR0 = 0.001                
LRF = 0.01                 
WEIGHT_DECAY = 0.001
CLS_GAIN = 2.0             
BOX_GAIN = 7.5             
CLOSE_MOSAIC = 15

# Add Robust augmentations for better test-set generalization
MIXUP = 0.15
COPY_PASTE = 0.1
DEGREES = 15.0

MIN_BOX_PX = 4
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SPLITS = ("train", "val", "test")
RUN_NAME = f"yolo11s_{IMG_SIZE}_v5_tuned"   
RUN_DIR = RUNS_DIR / RUN_NAME

assert SOURCE_DIR.exists(), "Dataset not found. Use Add Input, then copy the path with 'Copy file path' into SOURCE_DIR."
print("dataset contents:", sorted(p.name for p in SOURCE_DIR.iterdir()))

In [ ]:
# copying the dataset to a writable folder and reading the class names
shutil.rmtree(DATA_DIR, ignore_errors=True)
shutil.copytree(SOURCE_DIR, DATA_DIR)
for cache_file in DATA_DIR.rglob("*.cache"):
    cache_file.unlink()

original = yaml.safe_load((SOURCE_DIR / "data.yaml").read_text())
raw_names = original["names"]
NAMES = {int(k): v for k, v in (raw_names.items() if isinstance(raw_names, dict) else enumerate(raw_names))}
CLASS_LIST = [NAMES[i] for i in sorted(NAMES)]
REWARDED = {i for i, name in NAMES.items() if name in {"can", "plastic"}}

DATA_YAML.write_text(yaml.safe_dump({
    "path": str(DATA_DIR), "train": "train/images", "val": "val/images", "test": "test/images",
    "nc": len(NAMES), "names": CLASS_LIST,
}, sort_keys=False))
print("classes:", NAMES, "| rewarded ids:", sorted(REWARDED))

In [ ]:
# defining file-name helpers and counting the photos in each split
def group_of(name):
    match = re.match(r"(own_[BSE]\d+|pub_[A-Za-z0-9]+)", name)
    return match.group(1) if match else "unknown"


def kind_of(name):
    group = group_of(name)
    if group.startswith("own_B"):
        return "own_bulk"
    if group.startswith("own_S"):
        return "own_single"
    if group.startswith("own_E"):
        return "own_empty"
    if group.startswith("pub_"):
        return "public"
    return "unknown"


def list_images(split):
    return sorted(p for p in (DATA_DIR / split / "images").iterdir() if p.suffix.lower() in IMG_EXT)


for split in SPLITS:
    images = list_images(split)
    labels = list((DATA_DIR / split / "labels").glob("*.txt"))
    unrecognised = sum(kind_of(p.name) == "unknown" for p in images)
    print(f"{split:5s} images: {len(images):5d}   label files: {len(labels):5d}   unrecognised names: {unrecognised}")

In [ ]:
# removing accidental tiny boxes from the working copy and logging every change
changes = []
for split in SPLITS:
    for img in list_images(split):
        label = DATA_DIR / split / "labels" / (img.stem + ".txt")
        if not label.exists():
            continue
        with PILImage.open(img) as photo:
            width, height = photo.size
        kept, dropped = [], False
        for line in label.read_text().splitlines():
            parts = line.split()
            if len(parts) == 5 and (float(parts[3]) * width < MIN_BOX_PX or float(parts[4]) * height < MIN_BOX_PX):
                size = f"{float(parts[3]) * width:.1f}x{float(parts[4]) * height:.1f}px"
                changes.append(dict(split=split, image=img.name, box=f"{NAMES[int(float(parts[0]))]} {size}"))
                dropped = True
            elif line.strip():
                kept.append(line)
        if dropped:
            label.write_text("\n".join(kept) + ("\n" if kept else ""))

changes_df = pd.DataFrame(changes, columns=["split", "image", "box"])
changes_df.to_csv(CHANGE_LOG, index=False)
print(f"removed {len(changes_df)} tiny boxes, logged to {CHANGE_LOG}")
print(changes_df.to_string(index=False) if len(changes_df) else "nothing to remove")

In [ ]:
# reading every image and label file to build the audit tables
image_rows, box_rows = [], []
problems = dict(missing_label=0, not_5_columns=0, bad_class=0, out_of_range=0, tiny_box=0, duplicate_line=0)
file_hashes = {}

for split in SPLITS:
    for img in list_images(split):
        with PILImage.open(img) as photo:
            width, height = photo.size
        file_hashes.setdefault(hashlib.md5(img.read_bytes()).hexdigest(), []).append(split)
        label_file = DATA_DIR / split / "labels" / (img.stem + ".txt")
        if label_file.exists():
            rows = [line.split() for line in label_file.read_text().splitlines() if line.strip()]
        else:
            rows = []
            problems["missing_label"] += 1
        seen, kept = set(), 0
        for parts in rows:
            if len(parts) != 5:
                problems["not_5_columns"] += 1
                continue
            cls = int(float(parts[0]))
            cx, cy, bw, bh = map(float, parts[1:])
            if cls not in NAMES:
                problems["bad_class"] += 1
            if min(cx, cy, bw, bh) < 0 or max(cx, cy, bw, bh) > 1:
                problems["out_of_range"] += 1
            if bw * width < MIN_BOX_PX or bh * height < MIN_BOX_PX:
                problems["tiny_box"] += 1
            if tuple(parts) in seen:
                problems["duplicate_line"] += 1
            seen.add(tuple(parts))
            side = np.sqrt(bw * width * bh * height)
            box_rows.append(dict(split=split, kind=kind_of(img.name), group=group_of(img.name),
                                 cls=NAMES.get(cls, cls),
                                 side_640=side * 640 / max(width, height),
                                 side_1280=side * 1280 / max(width, height)))
            kept += 1
        image_rows.append(dict(split=split, kind=kind_of(img.name), group=group_of(img.name),
                               boxes=kept, path=str(img)))

images_df, boxes_df = pd.DataFrame(image_rows), pd.DataFrame(box_rows)
print(f"read {len(images_df)} images and {len(boxes_df)} boxes")

In [ ]:
# printing photo counts, class counts and crowd density
print("1) Images and objects, by split and kind")
by_kind = images_df.groupby(["split", "kind"]).agg(
    images=("boxes", "size"), empty=("boxes", lambda s: int((s == 0).sum())),
    objects=("boxes", "sum"), mean_per_image=("boxes", "mean"), max_per_image=("boxes", "max")).round(1)
print(by_kind.to_string())

print("\n2) Objects per class, by split")
print(boxes_df.pivot_table(index="split", columns="cls", values="side_640", aggfunc="size", fill_value=0).to_string())

print("\n3) Crowd scenes: how crowded is each one?")
bulk = images_df[images_df.kind == "own_bulk"].groupby(["split", "group"]).agg(
    images=("boxes", "size"), objects=("boxes", "sum"), mean_per_image=("boxes", "mean")).round(1)
print(bulk.to_string())

In [ ]:
# counting objects per class in every own-photo session
rows = []
for split in SPLITS:
    for img in list_images(split):
        if kind_of(img.name) == "public":
            continue
        label_file = DATA_DIR / split / "labels" / (img.stem + ".txt")
        lines = [l.split() for l in label_file.read_text().splitlines() if l.strip()] if label_file.exists() else []
        if not lines:
            rows.append(dict(split=split, group=group_of(img.name), image=img.name, cls="(empty image)"))
        for parts in lines:
            if len(parts) == 5:
                rows.append(dict(split=split, group=group_of(img.name), image=img.name, cls=NAMES[int(float(parts[0]))]))

own_df = pd.DataFrame(rows)
counts = own_df.pivot_table(index=["split", "group"], columns="cls", values="image", aggfunc="size", fill_value=0)
counts["photos"] = own_df.groupby(["split", "group"])["image"].nunique()
counts = counts.drop(columns="(empty image)", errors="ignore")
print("Objects per class in every own-photo session")
print(counts.to_string())

crowd = counts[counts.index.get_level_values("group").str.startswith("own_B")].copy()
for c in CLASS_LIST:
    crowd[f"{c} %"] = (100 * crowd[c] / crowd[CLASS_LIST].sum(axis=1)).round(0)
print("\nCrowd sessions: class share (%)")
print(crowd[[f"{c} %" for c in CLASS_LIST]].to_string())

In [ ]:
# setting how many times each session is shown per epoch
GROUP_REPEAT = {
    "own_B01": 0.25,
    "own_B02": 4,
    "own_B04": 2,
    "own_B05": 3,
    "own_B06": 3,
    "own_B09": 1,
    "own_B10": 3,
    "own_B11": 2,
}
DEFAULT_BULK_REPEAT = 2
SINGLE_REPEAT = 2
EMPTY_REPEAT = 1
PUBLIC_REPEAT = 1

In [ ]:
# building the weighted training list and checking what the model will see
train_images = list_images("train")
rng = np.random.default_rng(SEED)


def repeat_for(name):
    kind = kind_of(name)
    if kind == "public":
        return PUBLIC_REPEAT
    if kind == "own_single":
        return SINGLE_REPEAT
    if kind == "own_empty":
        return EMPTY_REPEAT
    if kind == "own_bulk":
        return GROUP_REPEAT.get(group_of(name), DEFAULT_BULK_REPEAT)
    raise ValueError(f"training file matches no naming pattern: {name}")


by_group = {}
for path in train_images:
    by_group.setdefault(group_of(path.name), []).append(path)

lines = []
for group, paths in sorted(by_group.items()):
    factor = repeat_for(paths[0].name)
    whole = int(np.floor(factor))
    extra = int(round((factor - whole) * len(paths)))
    lines += [str(p) for p in paths for _ in range(whole)]
    if extra:
        lines += [str(paths[i]) for i in sorted(rng.choice(len(paths), size=extra, replace=False))]

TRAIN_LIST.write_text("\n".join(lines) + "\n")
TRAIN_YAML.write_text(yaml.safe_dump({
    "path": str(DATA_DIR), "train": str(TRAIN_LIST), "val": "val/images", "test": "test/images",
    "nc": len(NAMES), "names": CLASS_LIST,
}, sort_keys=False))

listed = Counter(lines)
exposure = []
for path_str, times in listed.items():
    path = Path(path_str)
    label_file = DATA_DIR / "train" / "labels" / (path.stem + ".txt")
    rows = [l.split() for l in label_file.read_text().splitlines() if l.strip()] if label_file.exists() else []
    for row in rows:
        if len(row) == 5:
            exposure.append(dict(kind=kind_of(path.name), cls=NAMES[int(float(row[0]))], n=times))
exposure = pd.DataFrame(exposure)

own_share = sum(1 for l in lines if kind_of(Path(l).name) != "public") / len(lines)
unweighted_share = sum(1 for p in train_images if kind_of(p.name) != "public") / len(train_images)
print(f"unique training photos: {len(train_images)}   shown per epoch: {len(lines)}")
print(f"own photos are {own_share:.0%} of what the model sees (unweighted: {unweighted_share:.0%})")


def share(series):
    return (100 * series / series.sum()).round(1)


crowd_before = boxes_df[(boxes_df.split == "train") & (boxes_df.kind == "own_bulk")].groupby("cls").size()
crowd_after = exposure[exposure.kind == "own_bulk"].groupby("cls")["n"].sum()
crowd_test = boxes_df[(boxes_df.split == "test") & (boxes_df.kind == "own_bulk")].groupby("cls").size()
print("\nCrowd scenes: class share of objects (%)")
print(pd.DataFrame({"train before weighting": share(crowd_before), "train after weighting": share(crowd_after),
                    "test": share(crowd_test)}).to_string())
print("\nUltralytics should report this many training images when it scans:", len(lines))

In [ ]:
# drawing the labelled boxes on sample photos to check the labels by eye
CLASS_COLOURS = {"can": "red", "other": "yellow", "plastic": "cyan"}


def draw_labels(image_path, split, max_side=900):
    image = ImageOps.exif_transpose(PILImage.open(image_path)).convert("RGB")
    scale = max_side / max(image.size)
    image = image.resize((max(1, int(image.width * scale)), max(1, int(image.height * scale))))
    canvas = ImageDraw.Draw(image)
    label_file = DATA_DIR / split / "labels" / (Path(image_path).stem + ".txt")
    count = 0
    if label_file.exists():
        for line in label_file.read_text().splitlines():
            parts = line.split()
            if len(parts) != 5:
                continue
            cls = int(float(parts[0]))
            cx, cy, bw, bh = map(float, parts[1:])
            box = [(cx - bw / 2) * image.width, (cy - bh / 2) * image.height,
                   (cx + bw / 2) * image.width, (cy + bh / 2) * image.height]
            canvas.rectangle(box, outline=CLASS_COLOURS.get(NAMES.get(cls), "white"), width=2)
            count += 1
    return image, count


rng = np.random.default_rng()
wanted = [("train", "own_B10"), ("train", "own_B11"), ("val", "own_B12"), ("train", "own_B04"),
          ("train", "own_B05"), ("test", "own_B03"), ("val", "own_B07"), ("train", "pub_trashnet")]
chosen = []
for split_name, group_name in wanted:
    pool = images_df[(images_df.split == split_name) & (images_df.group == group_name) & (images_df.boxes > 0)]
    if len(pool):
        row = pool.iloc[int(rng.integers(len(pool)))]
        chosen.append((split_name, group_name, row.path))

figure, axes = plt.subplots(2, 4, figsize=(22, 11))
for axis, (split_name, group_name, path) in zip(axes.ravel(), chosen):
    picture, count = draw_labels(path, split_name)
    axis.imshow(picture)
    axis.set_title(f"{split_name} | {group_name} | {count} boxes", fontsize=11)
    axis.axis("off")
for axis in axes.ravel()[len(chosen):]:
    axis.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# saving the run settings so the experiment can be reproduced
RUN_DIR.mkdir(parents=True, exist_ok=True)
run_config = {
    "model": MODEL, "imgsz": IMG_SIZE, "batch": BATCH_SIZE, "epochs": EPOCHS, "patience": PATIENCE,
    "scale": SCALE, "max_det": MAX_DET, "seed": SEED, "cos_lr": True, "close_mosaic": CLOSE_MOSAIC,
    "freeze": FREEZE, "optimizer": "AdamW", "lr0": LR0, "weight_decay": WEIGHT_DECAY, "cls": CLS_GAIN,
    "classes": CLASS_LIST, "min_box_px": MIN_BOX_PX,
    "group_repeat": GROUP_REPEAT, "default_bulk_repeat": DEFAULT_BULK_REPEAT, "single_repeat": SINGLE_REPEAT,
    "empty_repeat": EMPTY_REPEAT, "public_repeat": PUBLIC_REPEAT,
    "images_per_split": {s: len(list_images(s)) for s in SPLITS},
    "train_images_listed_per_epoch": len(lines),
    "ultralytics": ultralytics.__version__, "torch": torch.__version__,
    "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
}
(RUN_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2))
shutil.copy2(CHANGE_LOG, RUN_DIR / "dataset_changes.csv")
shutil.copy2(TRAIN_LIST, RUN_DIR / "train_weighted.txt")
print(json.dumps(run_config, indent=2))

In [ ]:
# training the model
model = YOLO(MODEL)
results = model.train(
    data=str(TRAIN_YAML),
    epochs=EPOCHS,
    patience=PATIENCE,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=WORKERS,
    seed=SEED,
    cache="ram",
    cos_lr=True,
    close_mosaic=CLOSE_MOSAIC,
    scale=SCALE,
    mixup=MIXUP,               
    copy_paste=COPY_PASTE,     
    degrees=DEGREES,           
    freeze=FREEZE,
    optimizer="AdamW",
    lr0=LR0,
    lrf=LRF,                   
    weight_decay=WEIGHT_DECAY,
    cls=CLS_GAIN,
    box=BOX_GAIN,              
    max_det=MAX_DET,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
)

In [ ]:
# packaging the key training results into one small folder and zip
import zipfile

run = RUN_DIR
print("run folder exists:", run.exists())
keep = ["weights/best.pt", "weights/last.pt", "results.csv", "results.png", "confusion_matrix.png",
        "confusion_matrix_normalized.png", "BoxPR_curve.png", "BoxF1_curve.png", "labels.jpg",
        "args.yaml", "run_config.json", "dataset_changes.csv", "train_weighted.txt"]
keep += [p.name for p in sorted(run.glob("val_batch*_pred.jpg"))[:3]]
export = Path("/kaggle/working/export_v4")
shutil.rmtree(export, ignore_errors=True)
export.mkdir()
for name in keep:
    if (run / name).exists():
        shutil.copy2(run / name, export / Path(name).name)
zip_path = Path("/kaggle/working/export_v4.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for item in sorted(export.iterdir()):
        archive.write(item, item.name)
print({p.name: round(p.stat().st_size / 1e6, 1) for p in sorted(export.iterdir())})
print("zip size (MB):", round(zip_path.stat().st_size / 1e6, 1))

In [ ]:
# removing the dataset copy so the saved output stays small
shutil.rmtree(DATA_DIR, ignore_errors=True)
print("kept in /kaggle/working:", sorted(p.name for p in Path("/kaggle/working").iterdir()))